### 構造生成

ランダム変異を加えて構造を生成する。


In [1]:
import sys
import os
import numpy as np

from pymatgen.core import Lattice, Structure
from pymatgen.io.cif import CifWriter, CifParser
from pymatgen.io.xcrysden import XSF


In [2]:
np.random.seed(0)

In [3]:
g_path_prefix = "../data/Fe_structure"
g_files = [
    'Fe-bcc.cif',
    'Fe-fcc.cif',
    'Fe-hcp.cif'
]

In [4]:
def modify_cell(structure, noise_abc=0.02, noise_angle=0.03, noise_cart=0.05):
    """add small displacement, distortion to the structure

    Args:
        structure (Structure): crystal structure
        noise_abc (float, optional): noise to abc length. Defaults to 0.02.
        noise_angle (float, optional): noise to angles. Defaults to 0.02.
        nise_cart (float, optional): noise to cartesian coordinates. Defaults to 0.1.
        
    Returns:
        Structure: noised Structure
    """
    noise_abc_ = np.random.normal(loc=0.0, scale=noise_abc, size=3)   # N(0.0, noise_abc^2)
    noise_angle_ = np.random.normal(loc=0.0, scale=noise_angle, size=3) # N(0.0, noise_angle^2)
    noise_cart_ = np.random.random(size=1)*noise_cart  # uniform random value of [0, noise_cart]
    print("noise, abc, angle, cart", noise_abc_, noise_angle_, noise_cart_)
    
    abc = structure.lattice.abc
    abc += abc*noise_abc_

    angles = structure.lattice.angles
    angles += angles*noise_angle_

    print("abc", abc, "angles", angles)

    # lattice = Lattice.from_lengths_and_angles(abc, angles)
    lattice = Lattice.from_parameters(a=abc[0], b=abc[1], c=abc[2],
                                      alpha=angles[0], beta=angles[1], gamma=angles[2])
    
    distorted_structure = Structure(lattice, structure.species, structure.frac_coords)
    
    # add noise to cart_coords
    r = (np.random.random(size=distorted_structure.cart_coords.shape)-0.5)*noise_cart_
    cart_coords = distorted_structure.cart_coords + r
    
    struct = Structure(lattice, distorted_structure.species, cart_coords, coords_are_cartesian=True)
    
    return struct

copy original structures

In [5]:
import shutil
g_random_dir = "Fe_random_calculated"
os.makedirs(g_random_dir, exist_ok=True)

for _name in g_files:
    _inputfile = os.path.join(g_path_prefix, _name)
    _outputfile = os.path.join(g_random_dir, _name)
    shutil.copyfile(_inputfile, _outputfile)

In [6]:
def make_distorted_supercell(files, random_dir, nstructure: int = 20):
    """構造ファイルを読み、supercellをつくり、distortion, displacementを加える。

    Args:
        files ([str]): filenames
        random_dir (str): directory name to place the structures
        nstructure (int, optional): the number of structures. Defaults to 20.
    """
    for name in files:
        print()
        print(name)
        print()
        s = name.split(".")
        key = s[0]
        prototype = s[0].split("-")[0]
        if prototype=="hcp":
            supercell=[2,2,1]
        else:
            supercell=[2,2,2]
        cifparser = CifParser(os.path.join(random_dir,name))
        structure = cifparser.get_structures(primitive=False)[0]
        structure.make_supercell(supercell)
        for i in range(nstructure):
            struc = modify_cell(structure)
            name2 = "{}/{}_{:02}.cif".format(random_dir, key, i)
            print(name2)
            struc.to(fmt="cif", filename=name2)
            i += 1
            
g_nstructure = 15
make_distorted_supercell(g_files, g_random_dir, nstructure=g_nstructure)


Fe-bcc.cif

noise, abc, angle, cart [0.03528105 0.00800314 0.01957476] [ 0.0672268   0.05602674 -0.02931834] [0.02187936]
abc [5.93423096 5.77787402 5.84420252] angles [96.05041164 95.04240657 87.36134972]
Fe_random_calculated/Fe-bcc_00.cif
noise, abc, angle, cart [ 0.00897275 -0.0198486  -0.00451879] [-0.04963712 -0.01919168 -0.01438259] [0.00551876]
abc [5.78343181 5.61822785 5.70609828] angles [85.53265893 88.27274888 88.70556654]
Fe_random_calculated/Fe-bcc_01.cif
noise, abc, angle, cart [ 0.02357559 -0.0035985  -0.02141505] [ 0.03163355 -0.01209531  0.03667335] [0.04405516]
abc [5.86713529 5.71137342 5.60924892] angles [92.84701966 88.91142224 93.30060169]
Fe_random_calculated/Fe-bcc_02.cif
noise, abc, angle, cart [ 0.02687909 -0.01636442  0.0016542 ] [-0.03873175 -0.01983313 -0.03540573] [0.04640406]
abc [5.88607093 5.63819915 5.74148187] angles [86.51414217 88.21501857 86.81348426]
Fe_random_calculated/Fe-bcc_03.cif
noise, abc, angle, cart [-0.02068486  0.01363189 -0.01606819] 

/tmp/mi-in-place.WyFyuX/ipykernel_10895/875786502.py:21: FutureWarning: get_structures is deprecated; use parse_structures in pymatgen.io.cif instead.
The only difference is that primitive defaults to False in the new parse_structures method.So parse_structures(primitive=True) is equivalent to the old behavior of get_structures().
  structure = cifparser.get_structures(primitive=False)[0]
/home/kino/miniforge3/envs/pymc_env/lib/python3.14/site-packages/pymatgen/io/cif.py:1394: UserWarning: Issues encountered while parsing CIF: Skipping relative stoichiometry check because CIF does not contain formula keys.
  return self.parse_structures(*args, **kwargs)
/home/kino/miniforge3/envs/pymc_env/lib/python3.14/site-packages/pymatgen/io/cif.py:1394: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
Skipping relative stoichiometry check because CIF does not contain formula keys.
  return self.parse_structur

In [7]:
import json
with open("condition.json","w") as f:
    json.dump({"nstructure":g_nstructure},f)


In [8]:
"done"

'done'